In [1]:
import numpy as np
import pandas as pd

# 1. Read the combined environmental dataset directly from the CSV file
env_df = pd.read_csv(r"C:\Users\DELL\Downloads\Manhattan_Community_Districts_Environmental_Data (1).csv")

# Display the first few rows to verify
display(env_df.head())

# Clean/convert Tree Canopy Coverage column from string percentage (e.g. '14.86%') to float (14.86)
env_df["Tree_Canopy_Pct"] = (
    env_df["Tree_Canopy_Coverage_2021"]
    .astype(str)
    .str.rstrip("%")
    .astype(float)
)
# Rename columns to standardized pipeline names
env_df = env_df.rename(
    columns={
        "District_ID": "District_ID",
        "Community_District_Area": "Community_District_Area",
        "HVI_Score_2023": "HVI",
        "PM2.5_Annual_Mean_ug_m3_2024": "PM25",
    }
)

# 2. Add real district median ENERGY STAR scores from your Local Law 84 analysis
energy_star_dict = {
    "MN01": 65.0,
    "MN02": 66.0,
    "MN03": 59.0,
    "MN04": 71.0,
    "MN05": 72.0,
    "MN06": 70.0,
    "MN07": 78.0,
    "MN08": 70.0,
    "MN09": 72.0,
    "MN10": 69.0,
    "MN11": 66.0,
    "MN12": 65.0,
}

env_df["Energy_Star_Median"] = env_df["District_ID"].map(energy_star_dict)

print("--- DATASET LOADED FROM CSV ---")
print(
    env_df[
        [
            "District_ID",
            "Community_District_Area",
            "HVI",
            "Tree_Canopy_Pct",
            "PM25",
            "Energy_Star_Median",
        ]
    ]
)

,District_ID,Community_District_Area,HVI_Score_2023,Tree_Canopy_Coverage_2021,PM2.5_Annual_Mean_ug_m3_2024
0,MN01,Financial District–Tribeca,1,14.86%,8.1
1,MN02,Greenwich Village–SoHo,1,14.36%,9.4
2,MN03,Lower East Side–Chinatown,2,22.15%,8.1
3,MN04,Chelsea–Hell's Kitchen,2,11.65%,8.8
4,MN05,Midtown–Flatiron–Union Square,1,4.22%,10.5


--- DATASET LOADED FROM CSV ---
   District_ID               Community_District_Area  HVI  Tree_Canopy_Pct  \
0         MN01            Financial District–Tribeca    1            14.86   
1         MN02                Greenwich Village–SoHo    1            14.36   
2         MN03             Lower East Side–Chinatown    2            22.15   
3         MN04                Chelsea–Hell's Kitchen    2            11.65   
4         MN05         Midtown–Flatiron–Union Square    1             4.22   
5         MN06              East Midtown–Murray Hill    1            18.31   
6         MN07                       Upper West Side    1            26.13   
7         MN08      Upper East Side–Roosevelt Island    1            19.12   
8         MN09  Morningside Heights–Hamilton Heights    3            28.58   
9         MN10                                Harlem    5            23.93   
10        MN11                           East Harlem    4            20.62   
11        MN12             Washi

In [2]:
def compute_green_score(df: pd.DataFrame) -> pd.DataFrame:
  """Normalizes HVI, Tree Canopy, PM2.5, and ENERGY STAR scores to a 0-100 scale

  and calculates the overall aggregate Green Score (25% equal weights).
  """
  df_out = df.copy()

  # 1. Heat Index Normalization (Inverted 1-5 scale mapped to 0-100)
  # HVI 1 -> 100, HVI 3 -> 50, HVI 5 -> 0
  df_out['Heat_Score'] = (1.0 - (df_out['HVI'] - 1.0) / 4.0) * 100.0

  # 2. Tree Canopy Normalization (Target Cap = 50%)
  # Scaled linearly against the 50% target ceiling
  df_out['Canopy_Score'] = (df_out['Tree_Canopy_Pct'] / 50.0) * 100.0
  df_out['Canopy_Score'] = df_out['Canopy_Score'].clip(upper=100.0)

  # 3. Air Quality PM2.5 Normalization (Inverted Min-Max Scaling)
  # Lower PM2.5 concentration is better (higher score)
  pm_min, pm_max = df_out['PM25'].min(), df_out['PM25'].max()
  df_out['Air_Score'] = (
      1.0 - (df_out['PM25'] - pm_min) / (pm_max - pm_min)
  ) * 100.0

  # 4. Building Energy Efficiency Score
  # Direct 1-100 score using District Median ENERGY STAR Scores
  df_out['Energy_Score'] = df_out['Energy_Star_Median']

  # 5. Aggregate Green Score (Equal Weighted Combination: 25% each)
  df_out['Green_Score'] = (
      0.25 * df_out['Heat_Score']
      + 0.25 * df_out['Canopy_Score']
      + 0.25 * df_out['Air_Score']
      + 0.25 * df_out['Energy_Score']
  )

  return df_out


# Execute normalization pipeline
scored_df = compute_green_score(env_df)

In [3]:
# Select key columns for display
display_cols = [
    "District_ID",
    "Community_District_Area",
    "Heat_Score",
    "Canopy_Score",
    "Air_Score",
    "Energy_Score",
    "Green_Score",
]

# Display full results table sorted by highest Green Score
final_results = scored_df[display_cols].sort_values(
    by="Green_Score", ascending=False
)
print("--- FINAL MANHATTAN COMMUNITY DISTRICT GREEN SCORES ---")
print(final_results.to_string(index=False))

# Print summary metrics
print("\n--- SUMMARY METRICS ---")
print(f"Mean Green Score: {scored_df['Green_Score'].mean():.2f}")
print(f"Median Green Score: {scored_df['Green_Score'].median():.2f}")
print(
    f"Highest Scoring District:"
    f" {scored_df.loc[scored_df['Green_Score'].idxmax()]['Community_District_Area']}"
    f" ({scored_df['Green_Score'].max():.2f})"
)
print(
    f"Lowest Scoring District:"
    f" {scored_df.loc[scored_df['Green_Score'].idxmin()]['Community_District_Area']}"
    f" ({scored_df['Green_Score'].min():.2f})"
)

# Export processed Green Score results to a new CSV file
final_results.to_csv("Manhattan_Green_Scores_Final.csv", index=False)

--- FINAL MANHATTAN COMMUNITY DISTRICT GREEN SCORES ---
District_ID              Community_District_Area  Heat_Score  Canopy_Score  Air_Score  Energy_Score  Green_Score
       MN07                      Upper West Side       100.0         52.26  94.871795          78.0    81.282949
       MN12            Washington Heights–Inwood        75.0         69.40 100.000000          65.0    77.350000
       MN08     Upper East Side–Roosevelt Island       100.0         38.24  87.179487          70.0    73.854872
       MN09 Morningside Heights–Hamilton Heights        50.0         57.16  94.871795          72.0    68.507949
       MN06             East Midtown–Murray Hill       100.0         36.62  53.846154          70.0    65.116538
       MN01           Financial District–Tribeca       100.0         29.72  61.538462          65.0    64.064615
       MN03            Lower East Side–Chinatown        75.0         44.30  61.538462          59.0    59.959615
       MN11                          Eas